In [ ]:
!pip install evaluate transformers accelerate
!pip install datasets==3.6.0
!pip install --upgrade wandb

In [ ]:
!wandb login --relogin "insert_your_wandb_token_here"

In [ ]:
import os
from datetime import datetime
current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Current time: {current_time_str}")
# %%
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from collections import Counter

import pandas as pd
import numpy as np
import torch
import wandb

from datasets import (
    load_dataset,
    Audio
    # load_from_disk,
    # DatasetDict,
    # concatenate_datasets,
)

# %%

from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    Wav2Vec2Config,
    AutoConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

from huggingface_hub import login

# import Hugging Face libraries
import evaluate

from sklearn.metrics import confusion_matrix, classification_report

# %%
# check if there GPU
print("Check if GPU available:")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.get_device_name(): {torch.cuda.get_device_name()}")

# %%
#model_id = "facebook/mms-300m"
model_id = "utter-project/mHuBERT-147"
#model_id = "facebook/wav2vec2-xls-r-300m"


# %%
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id,
    do_normalize=True,
    return_attention_mask=True,
)

# %%
dataset = load_dataset("badrex/nnti-dataset-full")

# %%
# check the strucutre of the dataset object
print(f"dataset['train']: {dataset['train']}")

# %%
# check the strucutre of one training sample (before decoding)
print(f"dataset['train'][0]: {dataset['train'][0]}")

# %%
# shuffle the dataset
train_ds = dataset['train'].shuffle(seed=42)
valid_ds = dataset['validation'].shuffle(seed=42)

# resample to 16kHz
train_ds = train_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))


# %%
# based on the model typel, set input features key
if model_id == "facebook/w2v-bert-2.0":
    input_features_key = "input_features"
else:
    input_features_key = "input_values"

# %%
max_duration = 7 # in seconds

# %%
# get the set of languages
LABELS = train_ds.unique('language')

sorted_labels = sorted(l.upper() for l in LABELS)
print(f"Languages: {sorted_labels}")

str_to_int = {
    s: i for i, s in enumerate(LABELS)
}


# %%
def preprocess_function(examples):

    audio_arrays = [x["array"] for x in examples["audio_filepath"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        truncation=True,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        return_attention_mask=True,
    )

    inputs["label"] = [str_to_int[x] for x in examples["language"]]

    # convert input_features_key contains numerical arrays
    inputs[input_features_key] = [
        np.array(x) for x in inputs[input_features_key]
    ]

    inputs["length"] = [len(f) for f in inputs[input_features_key]]

    return inputs

# %%
keep_cols = ['speaker_id', 'language']

# ## encode the train and valid splits

# %%
train_ds_encoded = train_ds.map(
    preprocess_function,
    remove_columns=[c for c in train_ds.column_names if c not in keep_cols],
    batched=True,
    batch_size=32,
    #num_proc=8,
)

# %%
valid_ds_encoded = valid_ds.map(
    preprocess_function,
    remove_columns=[c for c in valid_ds.column_names if c not in keep_cols],
    batched=True,
    batch_size=32,
    #num_proc=8,
)

# %%
int_to_str = {
    i: s for s, i in str_to_int.items()
}

num_labels = len(int_to_str)

# %%
config = AutoConfig.from_pretrained(model_id)

config.num_labels=num_labels
config.label2id=str_to_int
config.id2label=int_to_str

do_apply_dropout = True # change

# check if dropout is enabled
if do_apply_dropout:
    config.hidden_dropout = 0.1           # Dropout for hidden states
    config.attention_dropout = 0.1        # Dropout in attention layers
    config.activation_dropout = 0.1       # Dropout after activation functions
    config.feat_proj_dropout = 0.1

# %%
# spoken language ID (SLID) model
slid_model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    config=config,
)

slid_model.freeze_feature_encoder()

# %%
# create collator for padding
class AudioDataCollator:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # prepare the batch dict in the format expected by the feature extractor
        batch = {
            input_features_key: [f[input_features_key] for f in features],
            "attention_mask": [f["attention_mask"] for f in features]
        }

        # use the feature extractor's native padding
        batch = self.feature_extractor.pad(
            batch,
            padding=True,
            return_tensors="pt"
        )

        # add labels
        batch["labels"] = torch.tensor(
            [f["label"] for f in features],
            dtype=torch.long
        )

        return batch


# %%
data_collator = AudioDataCollator(feature_extractor)

# %%
batch_size = 16
gradient_accumulation_steps = 4
num_train_epochs = 15
lr = 1e-4

# %%
wandb.init(project="Indic-SLID", name=f"SLID_{model_id}_{lr}_{current_time_str}")

# %%
training_args = TrainingArguments(
    output_dir="./results", # added as this must be 1st arg in t-ver
    dataloader_num_workers=4,
    #group_by_length=False,
    #run_name='SLID_1',
    report_to="wandb",  # enable logging to W&B
    logging_steps=1,  # how often to log to W&B
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    learning_rate=lr,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    label_smoothing_factor=0.1,
    lr_scheduler_type='cosine',
    weight_decay=0.15,
    warmup_ratio=0.15,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
)

# %%
# load evaluation metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(
        predictions=predictions,
        references=eval_pred.label_ids
    )


# %%
trainer = Trainer(
    slid_model,
    training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=valid_ds_encoded,
    processing_class=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# %%
print("Train loop starting...")
trainer.train()

# %%
# push model to hub
# slid_model.push_to_hub(
#     "your-hf-account/indic-language-identification"
# )

# %%
print("Final evaluation starting...")
metrics = trainer.evaluate()
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

print("\n" + "="*50, flush=True)
print("Generating eval metrics", flush=True)
print("="*50, flush=True)

val_outputs = trainer.predict(valid_ds_encoded)
y_pred = np.argmax(val_outputs.predictions, axis=1)
y_true = val_outputs.label_ids

ordered_label_names = [int_to_str[i] for i in range(num_labels)]

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=ordered_label_names, columns=ordered_label_names)

print("\nConfusion matrix", flush=True)
print(cm_df.to_string(), flush=True)

print("\nClassification report", flush=True)
report = classification_report(y_true, y_pred, target_names=ordered_label_names, zero_division=0)
print(report, flush=True)

print("="*50 + "\n", flush=True)

# save model to disk
save_dir = "./indic-SLID/inprogress"
slid_model.save_pretrained(save_dir)

print("generating tsne plot")

# %%
# tsne visualization
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

slid_model.eval()
device = next(slid_model.parameters()).device

all_embeddings = []
all_speaker_ids = []

tsne_batch_size = 16
valid_loader = torch.utils.data.DataLoader(
    valid_ds_encoded,
    batch_size=tsne_batch_size,
    collate_fn=data_collator,
    shuffle=False,
)

speaker_id_list = valid_ds_encoded["speaker_id"]

sample_idx = 0
with torch.no_grad():
    for batch in valid_loader:
        input_values = batch[input_features_key].to(device)
        attention_mask = batch["attention_mask"].to(device)
        curr_batch_size = input_values.shape[0]

        # get last hidden state from base model
        base_model = slid_model.hubert if hasattr(slid_model, 'hubert') else slid_model.wav2vec2 if hasattr(slid_model, 'wav2vec2') else slid_model.mms
        outputs = base_model(input_values=input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        # mean pooling
        if hasattr(base_model, '_get_feature_vector_attention_mask'):
            feat_mask = base_model._get_feature_vector_attention_mask(
                hidden_states.shape[1], attention_mask
            )
        else:
            feat_mask = torch.ones(hidden_states.shape[:2], device=device)

        mask_expanded = feat_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        all_embeddings.append(pooled.cpu().numpy())

        batch_speakers = speaker_id_list[sample_idx : sample_idx + curr_batch_size]
        all_speaker_ids.extend(batch_speakers)
        sample_idx += curr_batch_size

all_embeddings = np.concatenate(all_embeddings, axis=0)

print(f"embeddings shape: {all_embeddings.shape}")

# run tsne
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

# plot colored by speaker
unique_speakers = sorted(set(all_speaker_ids))
speaker_to_idx = {s: i for i, s in enumerate(unique_speakers)}
speaker_indices = np.array([speaker_to_idx[s] for s in all_speaker_ids])

plt.figure(figsize=(14, 10))
plt.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=speaker_indices, cmap='hsv',
    alpha=0.5, s=15,
)
plt.title("t-SNE - Colored by Speaker (Baseline)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tsne_by_speaker.png"), dpi=150)
plt.show()

print("completed")